# OmniVoice Live Server
Run all cells to start the live voice server. Make sure GPU is enabled under Runtime → Change runtime type → T4 GPU.

In [ ]:
# Cell 1: Install dependencies
!pip install torch torchaudio -q
!pip install omnivoice -q
!pip install fastapi uvicorn pydub pyngrok -q
!apt-get install -y ffmpeg -q

In [ ]:
# Cell 2: Download reference voice from GitHub
import requests

url = 'https://raw.githubusercontent.com/zaksix6/omnivoice-clone/main/reference.wav'
r = requests.get(url)
with open('reference.wav', 'wb') as f:
    f.write(r.content)
print('Reference voice downloaded!')

In [ ]:
# Cell 3: Load OmniVoice model
import torch
from omnivoice import OmniVoice

print('Loading model...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
print('Model ready!')

In [ ]:
# Cell 4: Start FastAPI server and tunnel
import os
import io
import re
import threading
import requests
import soundfile as sf
from fastapi import FastAPI
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from google.colab import userdata

# Get GitHub token from Colab secrets
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_REPO = 'zaksix6/omnivoice-clone'
GITHUB_FILE = 'tunnel_url.txt'

def update_github_url(url):
    api_url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_FILE}'
    headers = {'Authorization': f'token {GITHUB_TOKEN}', 'Content-Type': 'application/json'}
    r = requests.get(api_url, headers=headers)
    sha = r.json().get('sha', '')
    import base64
    content = base64.b64encode(url.encode()).decode()
    data = {'message': 'Update tunnel URL', 'content': content, 'sha': sha}
    requests.put(api_url, headers=headers, json=data)
    print(f'Tunnel URL written to GitHub: {url}')

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class TextRequest(BaseModel):
    text: str

@app.post('/speak')
async def speak(req: TextRequest):
    try:
        audio = model.generate(text=req.text, ref_audio='reference.wav')
        buf = io.BytesIO()
        sf.write(buf, audio[0], 24000, format='WAV')
        buf.seek(0)
        return StreamingResponse(buf, media_type='audio/wav')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.get('/health')
async def health():
    return {'status': 'ok'}

# Start cloudflared tunnel
import subprocess
import time

!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared

tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for tunnel URL
tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'Tunnel URL: {tunnel_url}')
    update_github_url(tunnel_url)
else:
    print('Could not get tunnel URL')

# Start server
print('Starting server...')
uvicorn.run(app, host='0.0.0.0', port=8000)